In [45]:
#!/usr/bin/env python3
"""
Model Training Pipeline — EV Purchase Prediction
Trains 5 classifiers on Silver and Gold feature sets with:
  - Stratified 5-fold cross-validation (Phase 1)
  - Optuna hyperparameter tuning on the best model (Phase 2)
  - 80/20 held-out evaluation, curves, confusion matrices
  - Artifact persistence (best_model.pkl, model_metadata.json)

Usage:
    python model_training_pipeline.py
"""

import os, sys, json, logging, pickle, warnings, time
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from scipy.stats import ks_2samp

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, brier_score_loss,
    confusion_matrix, roc_curve, precision_recall_curve,
)
import shap

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")

In [46]:
RANDOM_SEED   = 40
N_CV_FOLDS    = 5
OPTUNA_TRIALS = 150
TEST_SIZE     = 0.20
THRESHOLD_DEF = 0.50
current_dir=os.getcwd()

BASE_DIR      = os.path.dirname(os.path.abspath(current_dir))
GOLD_TRAIN    = os.path.join(BASE_DIR, "data", "Gold layer", "train_engineered.csv")
ARTIFACTS_DIR = os.path.join(BASE_DIR, "main_scripts", "pipeline_artifacts")
FIGURES_DIR   = os.path.join(BASE_DIR, "main_scripts","Reports", "figures")
REPORT_PATH   = os.path.join(BASE_DIR, "main_scripts","Reports", "model_comparison_report.md")
MODEL_PATH    = os.path.join(ARTIFACTS_DIR, "best_model.pkl")
META_PATH     = os.path.join(ARTIFACTS_DIR, "model_metadata.json")

# Feature set definitions (column names; populated after data load)
SILVER_COLS = [
    "Age",
    "Annual_Income_USD",
      "Daily_Commute_km", 
      "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",

    "Environmental_Concern_Level",
     "Gender", "City_Type", "Current_Car_Type",
    "Home_Charging_Possible", 
    "Subsidy_Available", "Range_Anxiety_Level",
    #New columns
    "income_exact","income_100","income_1000","Daily_Commute_km_exact"


]

ENGINEERED_COLS = [
    "flag_env_high_x_subsidy", "flag_env_low_no_subsidy",
    "flag_subsidy_x_low_anxiety", "flag_high_concern_low_anxiety",
    "flag_commute_sweet_spot", "flag_commute_long_haul",
    "income_segment", "income_x_anxiety",
    "env_x_income_product", "ev_propensity_score",
]

GOLD_COLS = SILVER_COLS + ENGINEERED_COLS

TARGET = "will_buy_ev"

FEATURE_SETS = {
    "silver": SILVER_COLS,
    "gold":   GOLD_COLS,
}

MODEL_NAMES = [
    "LogReg",  "RandomForest"
    ,"LightGBM", "XGBoost"
    , "CatBoost"
    ]

TREE_MODELS = {"RandomForest", "LightGBM", "XGBoost", "CatBoost"}


# Functions

In [47]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. LOGGING
# ══════════════════════════════════════════════════════════════════════════════

def setup_logger() -> logging.Logger:
    """
    Create and configure the pipeline logger.

    Returns:
        logging.Logger: Logger named 'pipeline' that writes INFO+ messages
                        with timestamps to stdout.
    """
    log = logging.getLogger("pipeline")
    log.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s  %(levelname)-7s  %(message)s", "%H:%M:%S")
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(fmt)
    log.addHandler(ch)
    return log

log = setup_logger()


# ══════════════════════════════════════════════════════════════════════════════
# 3. DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_gold(path: str):
    """
    Load the Gold-layer CSV into a DataFrame.

    Args:
        path (str): Absolute path to train_engineered.csv.

    Returns:
        pd.DataFrame: Full dataset including target column (will_buy_ev).
    """
    log.info(f"Loading Gold layer: {path}")
    df = pd.read_csv(path)
    log.info(f"  Shape: {df.shape}  |  Event rate: {df[TARGET].mean():.3f}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# 4. METRICS UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def compute_metrics(y_true, y_prob, threshold=THRESHOLD_DEF) -> dict:
    """
    Compute a full set of binary classification metrics.

    Args:
        y_true (array-like): Ground-truth binary labels (0/1).
        y_prob (array-like): Predicted probabilities for the positive class.
        threshold (float): Decision threshold for converting probabilities to
                           class labels. Defaults to THRESHOLD_DEF (0.50).

    Returns:
        dict: Keys — auc_roc, auc_pr, f1, precision, recall, brier, ks.
    """
    y_pred = (y_prob >= threshold).astype(int)
    pos_scores = y_prob[y_true == 1]
    neg_scores = y_prob[y_true == 0]
    ks_stat, _ = ks_2samp(pos_scores, neg_scores)
    return {
        "auc_roc":    roc_auc_score(y_true, y_prob),
        "auc_pr":     average_precision_score(y_true, y_prob),
        "f1":         f1_score(y_true, y_pred, zero_division=0),
        "precision":  precision_score(y_true, y_pred, zero_division=0),
        "recall":     recall_score(y_true, y_pred, zero_division=0),
        "brier":      brier_score_loss(y_true, y_prob),
        "ks":         ks_stat,
    }


def optimal_threshold(y_true, y_prob) -> float:
    """
    Find the probability threshold that maximises F1 score.

    Args:
        y_true (array-like): Ground-truth binary labels (0/1).
        y_prob (array-like): Predicted probabilities for the positive class.

    Returns:
        float: Threshold in [0.01, 0.99] that yields the highest F1.
    """
    thresholds = np.linspace(0.01, 0.99, 199)
    f1s = [f1_score(y_true, (y_prob >= t).astype(int), zero_division=0) for t in thresholds]
    return float(thresholds[np.argmax(f1s)])


def summarize_folds(fold_metrics: list[dict]) -> dict:
    """
    Aggregate per-fold metric dicts into mean and std across folds.

    Args:
        fold_metrics (list[dict]): One dict per fold, each with the same
                                   metric keys (output of compute_metrics).

    Returns:
        dict: {metric_name: {"mean": float, "std": float}} for every key.
    """
    keys = fold_metrics[0].keys()
    return {
        k: {
            "mean": float(np.mean([m[k] for m in fold_metrics])),
            "std":  float(np.std([m[k] for m in fold_metrics])),
        }
        for k in keys
    }


# ══════════════════════════════════════════════════════════════════════════════
# 5. MODEL DEFINITIONS
# ══════════════════════════════════════════════════════════════════════════════

def build_models(spw: float) -> dict:
    """
    Instantiate all 5 classifiers with default hyperparameters.

    Args:
        spw (float): scale_pos_weight = n_negatives / n_positives, passed to
                     gradient boosting models to handle class imbalance.
                     LogReg and RandomForest use class_weight='balanced' instead.

    Returns:
        dict: {model_name: unfitted sklearn-compatible classifier}.
    """
    return {
        "LogReg": LogisticRegression(
            max_iter=1000, class_weight="balanced",
            solver="saga", C=0.1, random_state=RANDOM_SEED,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300, max_depth=20, class_weight="balanced",
            n_jobs=-1, random_state=RANDOM_SEED,
        ),
        "LightGBM": lgb.LGBMClassifier(
            n_estimators=500, learning_rate=0.05, num_leaves=63,
            scale_pos_weight=spw, n_jobs=-1,
            random_state=RANDOM_SEED, verbosity=-1,
        ),
        "XGBoost": xgb.XGBClassifier(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            scale_pos_weight=spw, eval_metric="auc",
            n_jobs=-1, random_state=RANDOM_SEED, verbosity=0,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=500, learning_rate=0.05, depth=6,
            scale_pos_weight=spw,
            random_seed=RANDOM_SEED, verbose=0,
        ),
    }





# ══════════════════════════════════════════════════════════════════════════════
# 6. CROSS-VALIDATION RUNNER
# ══════════════════════════════════════════════════════════════════════════════

def cross_validate(model, X: pd.DataFrame, y: pd.Series) -> dict:
    """
    Run stratified K-fold cross-validation and return aggregated metrics.

    Args:
        model: Unfitted sklearn-compatible classifier with fit/predict_proba.
        X (pd.DataFrame): Feature matrix.
        y (pd.Series): Binary target (0/1).

    Returns:
        dict: {metric_name: {"mean": float, "std": float}} across N_CV_FOLDS folds.
    """
    skf = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    fold_metrics = []
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model.fit(X_tr, y_tr)
        y_prob = model.predict_proba(X_val)[:, 1]
        fold_metrics.append(compute_metrics(y_val, y_prob))
    return summarize_folds(fold_metrics)


# ══════════════════════════════════════════════════════════════════════════════
# 7. PHASE 1 — DEFAULT TRAINING
# ══════════════════════════════════════════════════════════════════════════════

def phase1(X_train: pd.DataFrame, y_train: pd.Series, spw: float) -> dict:
    """
    Run 5-fold CV for all 5 models across both feature sets (Silver and Gold).

    Args:
        X_train (pd.DataFrame): Training features (80% split), must contain
                                 all columns in SILVER_COLS and GOLD_COLS.
        y_train (pd.Series): Binary target for the training split.
        spw (float): scale_pos_weight passed to build_models.

    Returns:
        dict: Nested {feat_set: {model_name: {metric: {"mean", "std"}}}}
              for all combinations of feature sets and models.
    """
    log.info("=" * 60)
    log.info("PHASE 1 — 5-Fold Cross-Validation (default hyperparameters)")
    log.info("=" * 60)
    results = {}  # {feat_set: {model_name: {metric: {mean, std}}}}

    for fs_name, fs_cols in FEATURE_SETS.items():
        X = X_train[fs_cols]
        log.info(f"\n  Feature set: {fs_name.upper()}  ({len(fs_cols)} features)")
        results[fs_name] = {}
        models = build_models(spw)

        for name, model in models.items():
            t0 = time.time()
            log.info(f"    [{name}] running {N_CV_FOLDS}-fold CV ...")
            cv_result = cross_validate(model, X, y_train)
            elapsed = time.time() - t0
            auc = cv_result["auc_roc"]["mean"]
            std = cv_result["auc_roc"]["std"]
            log.info(f"    [{name}] AUC-ROC {auc:.4f} ± {std:.4f}  ({elapsed:.0f}s)")
            results[fs_name][name] = cv_result

    return results


# ══════════════════════════════════════════════════════════════════════════════
# 8. PHASE 2 — OPTUNA TUNING
# ══════════════════════════════════════════════════════════════════════════════

def best_from_phase1(cv_results: dict) -> tuple[str, str]:
    """
    Identify the feature set and model with the highest Phase-1 mean AUC-ROC.

    Args:
        cv_results (dict): Output of phase1 — nested {feat_set: {model_name: metrics}}.

    Returns:
        tuple[str, str]: (feat_set, model_name) of the winning combination.
    """
    best_score, best_fs, best_model = -1.0, None, None
    for fs_name, models in cv_results.items():
        for name, metrics in models.items():
            score = metrics["auc_roc"]["mean"]
            if score > best_score:
                best_score, best_fs, best_model = score, fs_name, name
    log.info(f"\n  Best Phase-1 model: {best_model} on {best_fs.upper()}  (AUC {best_score:.4f})")
    return best_fs, best_model


def _lgb_objective(trial, X, y, spw):
    """
    Optuna objective for LightGBM — samples hyperparameters and returns 3-fold CV AUC.

    Args:
        trial: Optuna Trial object used to suggest hyperparameter values.
        X (pd.DataFrame): Training features.
        y (pd.Series): Binary target.
        spw (float): scale_pos_weight for class imbalance.

    Returns:
        float: Mean AUC-ROC across 3 stratified folds (maximised by Optuna).
    """
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 200, 1500),
        learning_rate     = trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        num_leaves        = trial.suggest_int("num_leaves", 20, 300),
        max_depth         = trial.suggest_int("max_depth", 3, 12),
        min_child_samples = trial.suggest_int("min_child_samples", 5, 100),
        subsample         = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha         = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        scale_pos_weight  = spw, n_jobs=-1,
        random_state=RANDOM_SEED, verbosity=-1,
    )
    model = lgb.LGBMClassifier(**params)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        y_prob = model.predict_proba(X.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[val_idx], y_prob))
    return np.mean(scores)


def _xgb_objective(trial, X, y, spw):
    """
    Optuna objective for XGBoost — samples hyperparameters and returns 3-fold CV AUC.

    Args:
        trial: Optuna Trial object used to suggest hyperparameter values.
        X (pd.DataFrame): Training features.
        y (pd.Series): Binary target.
        spw (float): scale_pos_weight for class imbalance.

    Returns:
        float: Mean AUC-ROC across 3 stratified folds (maximised by Optuna).
    """
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 200, 1500),
        learning_rate    = trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        max_depth        = trial.suggest_int("max_depth", 3, 10),
        min_child_weight = trial.suggest_int("min_child_weight", 1, 10),
        subsample        = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        gamma            = trial.suggest_float("gamma", 0, 5),
        reg_alpha        = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda       = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        scale_pos_weight = spw, eval_metric="auc", n_jobs=-1,
        random_state=RANDOM_SEED, verbosity=0,
    )
    model = xgb.XGBClassifier(**params)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        y_prob = model.predict_proba(X.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[val_idx], y_prob))
    return np.mean(scores)


def _catboost_objective(trial, X, y, spw):
    """
    Optuna objective for CatBoost — samples hyperparameters and returns 3-fold CV AUC.

    Args:
        trial: Optuna Trial object used to suggest hyperparameter values.
        X (pd.DataFrame): Training features.
        y (pd.Series): Binary target.
        spw (float): scale_pos_weight for class imbalance.

    Returns:
        float: Mean AUC-ROC across 3 stratified folds (maximised by Optuna).
    """
    params = dict(
        iterations       = trial.suggest_int("iterations", 200, 1500),
        learning_rate    = trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        depth            = trial.suggest_int("depth", 4, 10),
        l2_leaf_reg      = trial.suggest_float("l2_leaf_reg", 1.0, 20.0),
        subsample        = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bylevel= trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        scale_pos_weight = spw,
        random_seed=RANDOM_SEED, verbose=0,
    )
    model = CatBoostClassifier(**params)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        y_prob = model.predict_proba(X.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[val_idx], y_prob))
    return np.mean(scores)


def _rf_objective(trial, X, y):
    """
    Optuna objective for RandomForest — samples hyperparameters and returns 3-fold CV AUC.

    Args:
        trial: Optuna Trial object used to suggest hyperparameter values.
        X (pd.DataFrame): Training features.
        y (pd.Series): Binary target.

    Returns:
        float: Mean AUC-ROC across 3 stratified folds (maximised by Optuna).
    """
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 100, 800),
        max_depth        = trial.suggest_int("max_depth", 5, 40),
        min_samples_split= trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10),
        max_features     = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        class_weight="balanced", n_jobs=-1, random_state=RANDOM_SEED,
    )
    model = RandomForestClassifier(**params)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        y_prob = model.predict_proba(X.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[val_idx], y_prob))
    return np.mean(scores)


def _logreg_objective(trial, X, y):
    """
    Optuna objective for LogisticRegression — tunes the regularisation strength C.

    Args:
        trial: Optuna Trial object used to suggest hyperparameter values.
        X (pd.DataFrame): Training features.
        y (pd.Series): Binary target.

    Returns:
        float: Mean AUC-ROC across 3 stratified folds (maximised by Optuna).
    """
    C = trial.suggest_float("C", 1e-4, 100.0, log=True)
    model = LogisticRegression(
        C=C, max_iter=1000, class_weight="balanced",
        solver="saga", random_state=RANDOM_SEED,
    )
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        y_prob = model.predict_proba(X.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[val_idx], y_prob))
    return np.mean(scores)


OBJECTIVES = {
    "LightGBM":    lambda trial, X, y, spw: _lgb_objective(trial, X, y, spw),
    "XGBoost":     lambda trial, X, y, spw: _xgb_objective(trial, X, y, spw),
    "CatBoost":    lambda trial, X, y, spw: _catboost_objective(trial, X, y, spw),
    "RandomForest":lambda trial, X, y, spw: _rf_objective(trial, X, y),
    "LogReg":      lambda trial, X, y, spw: _logreg_objective(trial, X, y),
}


def phase2(best_fs: str, best_model_name: str,
           X_train: pd.DataFrame, y_train: pd.Series, spw: float) -> tuple:
    """
    Run Optuna hyperparameter tuning on the best Phase-1 model and rebuild it.

    Args:
        best_fs (str): Feature set name ('silver' or 'gold') of the best model.
        best_model_name (str): Model name (e.g. 'XGBoost') of the best model.
        X_train (pd.DataFrame): Training features (80% split).
        y_train (pd.Series): Binary target for the training split.
        spw (float): scale_pos_weight for gradient boosting objectives.

    Returns:
        tuple:
            tuned_model: Unfitted classifier instantiated with the best params.
            study: Completed Optuna Study object (used for history plots).
            best_params (dict): Hyperparameters from the best trial.
            best_value (float): AUC-ROC of the best trial (3-fold CV).
    """
    log.info("")
    log.info("=" * 60)
    log.info(f"PHASE 2 — Optuna tuning  [{best_model_name} / {best_fs.upper()}]  ({OPTUNA_TRIALS} trials)")
    log.info("=" * 60)

    X = X_train[FEATURE_SETS[best_fs]]
    objective_fn = OBJECTIVES[best_model_name]

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )
    study.optimize(
        lambda trial: objective_fn(trial, X, y_train, spw),
        n_trials=OPTUNA_TRIALS,
        show_progress_bar=False,
    )

    best_params = study.best_params
    best_value  = study.best_value
    log.info(f"  Best trial AUC-ROC: {best_value:.4f}")
    log.info(f"  Best params: {best_params}")

    # Rebuild best model with tuned params
    if best_model_name == "LightGBM":
        best_params.update({"scale_pos_weight": spw, "n_jobs": -1,
                            "random_state": RANDOM_SEED, "verbosity": -1})
        tuned_model = lgb.LGBMClassifier(**best_params)
    elif best_model_name == "XGBoost":
        best_params.update({"scale_pos_weight": spw, "eval_metric": "auc",
                            "n_jobs": -1, "random_state": RANDOM_SEED, "verbosity": 0})
        tuned_model = xgb.XGBClassifier(**best_params)
    elif best_model_name == "CatBoost":
        best_params.update({"scale_pos_weight": spw,
                            "random_seed": RANDOM_SEED, "verbose": 0})
        tuned_model = CatBoostClassifier(**best_params)
    elif best_model_name == "RandomForest":
        best_params.update({"class_weight": "balanced", "n_jobs": -1,
                            "random_state": RANDOM_SEED})
        tuned_model = RandomForestClassifier(**best_params)
    else:  # LogReg
        tuned_model = LogisticRegression(
            C=best_params["C"], max_iter=1000, class_weight="balanced",
            solver="saga", random_state=RANDOM_SEED,
        )

    return tuned_model, study, best_params, best_value

# ══════════════════════════════════════════════════════════════════════════════
# 9. FINAL HELD-OUT EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_on_holdout(models_dict: dict, X_test: pd.DataFrame,
                        y_test: pd.Series, best_fs: str,
                        best_model_name: str) -> dict:
    """
    Evaluate all fitted models on the 20% held-out set.

    Args:
        models_dict (dict): Nested {feat_set: {model_name: fitted_model}}.
        X_test (pd.DataFrame): Held-out features (20% split).
        y_test (pd.Series): Held-out binary target.
        best_fs (str): Feature set of the best model (used for logging only).
        best_model_name (str): Name of the best model (used for logging only).

    Returns:
        dict: Nested {feat_set: {model_name: {
                  "metrics": dict,
                  "y_prob": np.ndarray,
                  "opt_threshold": float,
                  "cm_default": np.ndarray,
                  "cm_optimal": np.ndarray
              }}}.
    """
    log.info("")
    log.info("=" * 60)
    log.info("FINAL HELD-OUT EVALUATION (20% hold-out set)")
    log.info("=" * 60)
    holdout = {}  # {fs_name: {model_name: {metrics, y_prob, y_test}}}

    for fs_name, fs_cols in FEATURE_SETS.items():
        X = X_test[fs_cols]
        holdout[fs_name] = {}
        for name, model in models_dict[fs_name].items():
            y_prob = model.predict_proba(X)[:, 1]
            metrics = compute_metrics(y_test, y_prob)
            opt_thr = optimal_threshold(y_test, y_prob)
            y_pred_opt = (y_prob >= opt_thr).astype(int)
            holdout[fs_name][name] = {
                "metrics":      metrics,
                "y_prob":       y_prob,
                "opt_threshold":opt_thr,
                "cm_default":   confusion_matrix(y_test, (y_prob >= THRESHOLD_DEF).astype(int)),
                "cm_optimal":   confusion_matrix(y_test, y_pred_opt),
            }
            log.info(f"  [{fs_name.upper()} / {name}]  AUC {metrics['auc_roc']:.4f}  "
                     f"F1 {metrics['f1']:.4f}  KS {metrics['ks']:.4f}")

    return holdout



In [73]:
# ══════════════════════════════════════════════════════════════════════════════
# 10. VISUALIZATIONS
# ══════════════════════════════════════════════════════════════════════════════

PALETTE = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd"]


def plot_roc(holdout: dict, y_test: pd.Series, out_dir: str) -> None:
    """
    Save ROC curve plots (one PNG per feature set) to out_dir.

    Args:
        holdout (dict): Output of evaluate_on_holdout.
        y_test (pd.Series): Held-out binary target.
        out_dir (str): Directory where PNGs are saved.

    Returns:
        None. Saves roc_curves_{fs_name}.png for each feature set.
    """
    for fs_name, fs_models in holdout.items():
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5, label="Random")
        for (name, data), color in zip(fs_models.items(), PALETTE):
            fpr, tpr, _ = roc_curve(y_test, data["y_prob"])
            auc = data["metrics"]["auc_roc"]
            ax.plot(fpr, tpr, color=color, lw=1.8, label=f"{name}  (AUC {auc:.4f})")
        ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
               title=f"ROC Curves — {fs_name.upper()} features")
        ax.legend(fontsize=9)
        fig.tight_layout()
        path = os.path.join(out_dir, f"roc_curves_{fs_name}.png")
        fig.savefig(path, dpi=150)
        plt.close(fig)
        log.info(f"  Saved: {path}")


def plot_pr(holdout: dict, y_test: pd.Series, out_dir: str) -> None:
    """
    Save Precision-Recall curve plots (one PNG per feature set) to out_dir.

    Args:
        holdout (dict): Output of evaluate_on_holdout.
        y_test (pd.Series): Held-out binary target.
        out_dir (str): Directory where PNGs are saved.

    Returns:
        None. Saves pr_curves_{fs_name}.png for each feature set.
    """
    baseline = y_test.mean()
    for fs_name, fs_models in holdout.items():
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.axhline(baseline, color="k", ls="--", lw=0.8, alpha=0.5,
                   label=f"Baseline ({baseline:.3f})")
        for (name, data), color in zip(fs_models.items(), PALETTE):
            prec, rec, _ = precision_recall_curve(y_test, data["y_prob"])
            ap = data["metrics"]["auc_pr"]
            ax.plot(rec, prec, color=color, lw=1.8, label=f"{name}  (AP {ap:.4f})")
        ax.set(xlabel="Recall", ylabel="Precision",
               title=f"Precision–Recall Curves — {fs_name.upper()} features")
        ax.legend(fontsize=9)
        fig.tight_layout()
        path = os.path.join(out_dir, f"pr_curves_{fs_name}.png")
        fig.savefig(path, dpi=150)
        plt.close(fig)
        log.info(f"  Saved: {path}")


def plot_feature_importance(model, model_name: str, fs_cols: list, out_dir: str) -> None:
    """
    Save a horizontal bar chart of the top-20 feature importances to out_dir.
    Skipped silently if the model has no feature_importances_ attribute (e.g. LogReg).

    Args:
        model: Fitted model with an optional feature_importances_ attribute.
        model_name (str): Label used in the plot title.
        fs_cols (list[str]): Feature names in the same order as the model was trained.
        out_dir (str): Directory where the PNG is saved.

    Returns:
        None. Saves feature_importance.png.
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    try:
        if hasattr(model, "feature_importances_"):
            importances = model.feature_importances_
            idx = np.argsort(importances)[-20:]
            ax.barh([fs_cols[i] for i in idx], importances[idx], color="#1f77b4")
            ax.set(xlabel="Feature Importance", title=f"Feature Importance — {model_name}")
            fig.tight_layout()
            path = os.path.join(out_dir, "feature_importance.png")
            fig.savefig(path, dpi=150)
            log.info(f"  Saved: {path}")
        else:
            log.info(f"  [{model_name}] no feature_importances_ attribute — skipping")
    except Exception as e:
        log.warning(f"  Feature importance plot failed: {e}")
    plt.close(fig)


def plot_confusion_matrix(cm: np.ndarray, title: str, path: str) -> None:
    """
    Save a 2×2 confusion matrix heatmap as a PNG.

    Args:
        cm (np.ndarray): 2×2 confusion matrix (output of sklearn confusion_matrix).
        title (str): Plot title, typically includes model name and threshold.
        path (str): Full file path for the saved PNG.

    Returns:
        None. Saves the PNG at path.
    """
    # Normalize each row so values sum to 100% (per true class)
    cm_norm = cm #.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    vmax=100
    vmax=cm_norm.sum()
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=vmax)
    ax.set(xticks=[0, 1], yticks=[0, 1],
           xticklabels=["Pred 0", "Pred 1"],
           yticklabels=["True 0", "True 1"],
           title=title)
    for i in range(2):
        for j in range(2):
            #ax.text(j, i, f"{cm_norm[i,j]:.1f}%", ha="center", va="center",
            #        color="white" if cm_norm[i,j] > 50 else "black", fontsize=12)
            ax.text(j, i, f"{cm_norm[i,j]}", ha="center", va="center",
                    color="white" if cm_norm[i,j] > 50 else "black", fontsize=12)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    log.info(f"  Saved: {path}")
    
def plot_confusion_matrix(cm: np.ndarray, title: str, path: str) -> None:
    """
    Save a 2×2 confusion matrix heatmap as a PNG.

    Args:
        cm (np.ndarray): 2×2 confusion matrix (output of sklearn confusion_matrix).
        title (str): Plot title, typically includes model name and threshold.
        path (str): Full file path for the saved PNG.

    Returns:
        None. Saves the PNG at path.
    """
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max())  # ✅ vmax = highest cell value
    ax.set(xticks=[0, 1], yticks=[0, 1],
           xticklabels=["Pred 0", "Pred 1"],
           yticklabels=["True 0", "True 1"],
           title=title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                    color="white" if cm[i,j] > cm.max() / 2 else "black",  # ✅ threshold relative to max
                    fontsize=12)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    log.info(f"  Saved: {path}")

def plot_optuna_history(study, out_dir: str) -> None:
    """
    Save a scatter+line plot of Optuna trial AUC values and the running best.

    Args:
        study: Completed Optuna Study object.
        out_dir (str): Directory where the PNG is saved.

    Returns:
        None. Saves optuna_history.png.
    """
    trials = [t for t in study.trials if t.value is not None]
    values = [t.value for t in trials]
    best_so_far = np.maximum.accumulate(values)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(range(len(values)), values, s=12, alpha=0.5, color="#1f77b4", label="Trial")
    ax.plot(range(len(best_so_far)), best_so_far, color="#d62728", lw=1.8, label="Best so far")
    ax.set(xlabel="Trial", ylabel="AUC-ROC (3-fold CV)",
           title="Optuna Optimization History")
    ax.legend()
    fig.tight_layout()
    path = os.path.join(out_dir, "optuna_history.png")
    fig.savefig(path, dpi=150)
    plt.close(fig)
    log.info(f"  Saved: {path}")

In [49]:
def plot_shap_beeswarm(
    shap_values: shap.Explanation,
    title: str,
    path: str,
    max_display: int = 20,
) -> None:
    """
    Save a SHAP beeswarm plot as a PNG.

    Args:
        shap_values (shap.Explanation): SHAP Explanation object output from
                                        a SHAP explainer (e.g. shap.TreeExplainer).
        title (str):                    Plot title, typically includes model name.
        path (str):                     Full file path for the saved PNG.
        max_display (int):              Maximum number of features to display.
                                        Defaults to 20.

    Returns:
        None. Saves the PNG at path.
    """
    # --- For binary classifiers, slice class 1 if needed ------------------
    if shap_values.values.ndim == 3:
        shap_values = shap_values[:, :, 1]

    # --- Plot --------------------------------------------------------------
    # SHAP beeswarm does NOT support ax= injection, so we:
    # 1. Pre-set the figure size via plt.gcf() before SHAP draws
    # 2. Let SHAP create its own figure internally (plot_size=None)
    # 3. Grab the active figure afterwards to add title & save

    plt.figure(figsize=(10, max_display * 0.4 + 2))

    shap.plots.beeswarm(
        shap_values,
        max_display=max_display,
        plot_size=None,   # disables SHAP's internal resizing, respects our figsize
        show=False,
    )

    fig = plt.gcf()
    fig.suptitle(title, fontsize=13, fontweight="bold", y=1.01)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    log.info(f"  Saved: {path}")

In [50]:
# ══════════════════════════════════════════════════════════════════════════════
# 11. REPORT GENERATION
# ══════════════════════════════════════════════════════════════════════════════

METRIC_LABELS = {
    "auc_roc":   "AUC-ROC ⭐",
    "auc_pr":    "AUC-PR",
    "f1":        "F1",
    "precision": "Precision",
    "recall":    "Recall",
    "brier":     "Brier ↓",
    "ks":        "KS",
}


def _cv_table(cv_results: dict, fs_name: str) -> str:
    """
    Render a Markdown table of Phase-1 CV results for one feature set.

    Args:
        cv_results (dict): Output of phase1.
        fs_name (str): Feature set key ('silver' or 'gold').

    Returns:
        str: Multi-line Markdown table string.
    """
    lines = [
        f"### {fs_name.upper()} features — 5-Fold CV",
        "",
        "| Model | AUC-ROC | AUC-PR | F1 | Precision | Recall | Brier ↓ | KS |",
        "|---|---|---|---|---|---|---|---|",
    ]
    for name in MODEL_NAMES:
        m = cv_results[fs_name][name]
        def _fmt(k):
            return f"{m[k]['mean']:.4f} ±{m[k]['std']:.4f}"
        lines.append(
            f"| {name} | {_fmt('auc_roc')} | {_fmt('auc_pr')} | {_fmt('f1')} "
            f"| {_fmt('precision')} | {_fmt('recall')} | {_fmt('brier')} | {_fmt('ks')} |"
        )
    return "\n".join(lines)


def _holdout_table(holdout: dict, fs_name: str) -> str:
    """
    Render a Markdown table of held-out evaluation results for one feature set.

    Args:
        holdout (dict): Output of evaluate_on_holdout.
        fs_name (str): Feature set key ('silver' or 'gold').

    Returns:
        str: Multi-line Markdown table string including optimal threshold column.
    """
    lines = [
        f"### {fs_name.upper()} features — Held-Out (20%)",
        "",
        "| Model | AUC-ROC | AUC-PR | F1 | Precision | Recall | Brier ↓ | KS | Opt. Threshold |",
        "|---|---|---|---|---|---|---|---|---|",
    ]
    for name in MODEL_NAMES:
        d = holdout[fs_name][name]
        m = d["metrics"]
        thr = d["opt_threshold"]
        lines.append(
            f"| {name} | {m['auc_roc']:.4f} | {m['auc_pr']:.4f} | {m['f1']:.4f} "
            f"| {m['precision']:.4f} | {m['recall']:.4f} | {m['brier']:.4f} "
            f"| {m['ks']:.4f} | {thr:.3f} |"
        )
    return "\n".join(lines)


def write_report(cv_results: dict, holdout: dict, best_fs: str, best_model_name: str,
                 tuned_auc: float, best_params: dict, path: str) -> None:
    """
    Write the full Markdown comparison report to disk.

    Args:
        cv_results (dict): Output of phase1.
        holdout (dict): Output of evaluate_on_holdout.
        best_fs (str): Feature set of the best model.
        best_model_name (str): Name of the best model.
        tuned_auc (float): Best AUC from Optuna (3-fold CV).
        best_params (dict): Tuned hyperparameters from Optuna.
        path (str): Full file path for the saved .md report.

    Returns:
        None. Saves the report at path.
    """
    ts = datetime.now().strftime("%Y-%m-%d %H:%M")
    sections = [
        f"# Model Comparison Report — EV Purchase Prediction",
        f"**Generated:** {ts}  |  **Seed:** {RANDOM_SEED}  |  **CV folds:** {N_CV_FOLDS}  "
        f"|  **Optuna trials:** {OPTUNA_TRIALS}",
        "",
        "---",
        "",
        "## Phase 1 — Cross-Validation Results",
        "",
    ]
    for fs in ["silver", "gold"]:
        sections.append(_cv_table(cv_results, fs))
        sections.append("")

    sections += [
        "---",
        "",
        "## Phase 2 — Best Model & Optuna Tuning",
        "",
        f"Best Phase-1 model: **{best_model_name}** on **{best_fs.upper()}** features",
        "",
        f"Optuna best AUC-ROC (3-fold, {OPTUNA_TRIALS} trials): **{tuned_auc:.4f}**",
        "",
        "**Tuned hyperparameters:**",
        "",
        "```json",
        json.dumps(best_params, indent=2),
        "```",
        "",
        "---",
        "",
        "## Held-Out Evaluation (20%)",
        "",
    ]
    for fs in ["silver", "gold"]:
        sections.append(_holdout_table(holdout, fs))
        sections.append("")

    sections += [
        "---",
        "",
        "## Figures",
        "",
        "| File | Description |",
        "|---|---|",
        "| `figures/roc_curves_silver.png` | ROC curves — Silver feature set |",
        "| `figures/roc_curves_gold.png` | ROC curves — Gold feature set |",
        "| `figures/pr_curves_silver.png` | PR curves — Silver feature set |",
        "| `figures/pr_curves_gold.png` | PR curves — Gold feature set |",
        "| `figures/feature_importance.png` | Feature importances — best tuned model |",
        "| `figures/confusion_matrix_default.png` | Confusion matrix at threshold 0.50 |",
        "| `figures/confusion_matrix_optimal.png` | Confusion matrix at optimal F1 threshold |",
        "| `figures/optuna_history.png` | Optuna trial history |",
    ]

    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write("\n".join(sections) + "\n")
    log.info(f"  Report saved → {path}")


In [51]:
# ══════════════════════════════════════════════════════════════════════════════
# 12. ARTIFACT PERSISTENCE
# ══════════════════════════════════════════════════════════════════════════════

def save_artifacts(tuned_model, best_model_name: str, best_fs: str,
                   best_params: dict, tuned_auc: float, holdout: dict) -> None:
    """
    Persist the tuned model and its metadata to ARTIFACTS_DIR.

    Args:
        tuned_model: Fitted tuned classifier to pickle.
        best_model_name (str): Model name stored in metadata.
        best_fs (str): Feature set name stored in metadata.
        best_params (dict): Tuned hyperparameters stored in metadata.
        tuned_auc (float): Optuna best AUC stored in metadata.
        holdout (dict): Output of evaluate_on_holdout — holdout metrics are
                        extracted for the best_fs / best_model_name entry.

    Returns:
        None. Writes best_model.pkl and model_metadata.json to ARTIFACTS_DIR.
    """
    os.makedirs(ARTIFACTS_DIR, exist_ok=True)

    with open(MODEL_PATH, "wb") as f:
        pickle.dump(tuned_model, f)
    log.info(f"  Model saved → {MODEL_PATH}")

    holdout_metrics = holdout[best_fs][best_model_name]["metrics"]
    metadata = {
        "model_name":      best_model_name,
        "feature_set":     best_fs,
        "feature_columns": FEATURE_SETS[best_fs],
        "random_seed":     RANDOM_SEED,
        "cv_folds":        N_CV_FOLDS,
        "optuna_trials":   OPTUNA_TRIALS,
        "best_params":     best_params,
        "tuned_cv_auc":    round(tuned_auc, 6),
        "holdout_metrics": {k: round(v, 6) for k, v in holdout_metrics.items()},
        "trained_at":      datetime.now().isoformat(),
    }
    with open(META_PATH, "w") as f:
        json.dump(metadata, f, indent=2)
    log.info(f"  Metadata saved → {META_PATH}")

# Main

In [52]:
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# ── Load data ─────────────────────────────────────────────────────────────
df = load_gold(GOLD_TRAIN)
y  = df[TARGET]
X  = df.drop(columns=[TARGET])

# Class imbalance ratio
neg = (y == 0).sum()
pos = (y == 1).sum()
spw = neg / pos
log.info(f"  scale_pos_weight: {spw:.4f}  (neg={neg:,}, pos={pos:,})")

21:14:50  INFO     Loading Gold layer: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/data/Gold layer/train_engineered.csv
21:14:50  INFO     Loading Gold layer: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/data/Gold layer/train_engineered.csv
21:14:50  INFO     Loading Gold layer: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/data/Gold layer/train_engineered.csv
21:14:50  INFO       Shape: (668665, 28)  |  Event rate: 0.175
21:14:50  INFO       Shape: (668665, 28)  |  Event rate: 0.175
21:14:50  INFO       Shape: (668665, 28)  |  Event rate: 0.175
21:14:51  INFO       scale_pos_weight: 4.7259  (neg=551,886, pos=116,779)
21:14:51  INFO       scale_pos_weight: 4.7259  (neg=551,886, pos=116,779)
21:14:51  INFO       scale_pos_weight: 4.7259  (neg=551,886, pos=116,779)


In [53]:
# ── 80/20 split ───────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_SEED
)
log.info(f"  Train: {X_train.shape}  |  Test: {X_test.shape}")

21:14:51  INFO       Train: (534932, 27)  |  Test: (133733, 27)
21:14:51  INFO       Train: (534932, 27)  |  Test: (133733, 27)
21:14:51  INFO       Train: (534932, 27)  |  Test: (133733, 27)


In [54]:
X_train.columns

Index(['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
       'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
       'Environmental_Concern_Level', 'Gender', 'City_Type',
       'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available',
       'Range_Anxiety_Level', 'income_exact', 'income_100', 'income_1000',
       'Daily_Commute_km_exact', 'flag_env_high_x_subsidy',
       'flag_env_low_no_subsidy', 'flag_subsidy_x_low_anxiety',
       'flag_high_concern_low_anxiety', 'flag_commute_sweet_spot',
       'flag_commute_long_haul', 'income_segment', 'income_x_anxiety',
       'env_x_income_product', 'ev_propensity_score'],
      dtype='object')

In [55]:
# ── Phase 1: CV on train split ────────────────────────────────────────────
cv_results = phase1(X_train, y_train, spw)

# ── Phase 2: Optuna on best model ─────────────────────────────────────────
best_fs, best_model_name = best_from_phase1(cv_results)
tuned_model, study, best_params, tuned_auc = phase2(
    best_fs, best_model_name, X_train, y_train, spw
    )

21:14:51  INFO     ============================================================
21:14:51  INFO     ============================================================
21:14:51  INFO     ============================================================
21:14:51  INFO     PHASE 1 — 5-Fold Cross-Validation (default hyperparameters)
21:14:51  INFO     PHASE 1 — 5-Fold Cross-Validation (default hyperparameters)
21:14:51  INFO     PHASE 1 — 5-Fold Cross-Validation (default hyperparameters)
21:14:51  INFO     ============================================================
21:14:51  INFO     ============================================================
21:14:51  INFO     ============================================================
21:14:51  INFO     
  Feature set: SILVER  (17 features)
21:14:51  INFO     
  Feature set: SILVER  (17 features)
21:14:51  INFO     
  Feature set: SILVER  (17 features)
21:14:51  INFO         [LogReg] running 5-fold CV ...
21:14:51  INFO         [LogReg] running 5-fold CV ...
21:1

In [56]:
best_model_name

'XGBoost'

In [57]:
# ── Train all default models on full train split (for held-out eval) ──────
log.info("")
log.info("Training all models on 80% split for held-out evaluation ...")
trained_models = {}
for fs_name, fs_cols in FEATURE_SETS.items():
    trained_models[fs_name] = {}
    models = build_models(spw)
    for name, model in models.items():
        model.fit(X_train[fs_cols], y_train)
        trained_models[fs_name][name] = model
        log.info(f"  [{fs_name.upper()} / {name}] fitted")


22:11:00  INFO     
22:11:00  INFO     
22:11:00  INFO     
22:11:00  INFO     Training all models on 80% split for held-out evaluation ...
22:11:00  INFO     Training all models on 80% split for held-out evaluation ...
22:11:00  INFO     Training all models on 80% split for held-out evaluation ...
22:12:41  INFO       [SILVER / LogReg] fitted
22:12:41  INFO       [SILVER / LogReg] fitted
22:12:41  INFO       [SILVER / LogReg] fitted
22:12:58  INFO       [SILVER / RandomForest] fitted
22:12:58  INFO       [SILVER / RandomForest] fitted
22:12:58  INFO       [SILVER / RandomForest] fitted
22:13:05  INFO       [SILVER / LightGBM] fitted
22:13:05  INFO       [SILVER / LightGBM] fitted
22:13:05  INFO       [SILVER / LightGBM] fitted
22:13:07  INFO       [SILVER / XGBoost] fitted
22:13:07  INFO       [SILVER / XGBoost] fitted
22:13:07  INFO       [SILVER / XGBoost] fitted
22:13:15  INFO       [SILVER / CatBoost] fitted
22:13:15  INFO       [SILVER / CatBoost] fitted
22:13:15  INFO       [SIL

In [58]:
# Replace best model entry with tuned version (fit on train split)
log.info(f"\n  Fitting TUNED {best_model_name} on {best_fs.upper()} ...")
tuned_model.fit(X_train[FEATURE_SETS[best_fs]], y_train)
trained_models[best_fs][best_model_name] = tuned_model

# ── Held-out evaluation ───────────────────────────────────────────────────
holdout = evaluate_on_holdout(trained_models, X_test, y_test, best_fs, best_model_name)

22:15:58  INFO     
  Fitting TUNED XGBoost on GOLD ...
22:15:58  INFO     
  Fitting TUNED XGBoost on GOLD ...
22:15:58  INFO     
  Fitting TUNED XGBoost on GOLD ...
22:16:02  INFO     
22:16:02  INFO     
22:16:02  INFO     
22:16:02  INFO     ============================================================
22:16:02  INFO     ============================================================
22:16:02  INFO     ============================================================
22:16:02  INFO     FINAL HELD-OUT EVALUATION (20% hold-out set)
22:16:02  INFO     FINAL HELD-OUT EVALUATION (20% hold-out set)
22:16:02  INFO     FINAL HELD-OUT EVALUATION (20% hold-out set)
22:16:02  INFO     ============================================================
22:16:02  INFO     ============================================================
22:16:02  INFO     ============================================================
22:16:03  INFO       [SILVER / LogReg]  AUC 0.6390  F1 0.3397  KS 0.2023
22:16:03  INFO       [SILVE

In [74]:
# ── Visualizations ────────────────────────────────────────────────────────
log.info("")
log.info("Generating visualizations ...")
plot_roc(holdout, y_test, FIGURES_DIR)
plot_pr(holdout, y_test, FIGURES_DIR)

best_data = holdout[best_fs][best_model_name]
plot_feature_importance(
    tuned_model, best_model_name, FEATURE_SETS[best_fs], FIGURES_DIR
)
plot_confusion_matrix(
    best_data["cm_default"],
    f"{best_model_name} — threshold {THRESHOLD_DEF}",
    os.path.join(FIGURES_DIR, "confusion_matrix_default.png"),
)
plot_confusion_matrix(
    best_data["cm_optimal"],
    f"{best_model_name} — optimal threshold {best_data['opt_threshold']:.3f}",
    os.path.join(FIGURES_DIR, "confusion_matrix_optimal.png"),
)
plot_optuna_history(study, FIGURES_DIR)

08:30:06  INFO     
08:30:06  INFO     
08:30:06  INFO     
08:30:06  INFO     Generating visualizations ...
08:30:06  INFO     Generating visualizations ...
08:30:06  INFO     Generating visualizations ...
08:30:07  INFO       Saved: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/figures/roc_curves_silver.png
08:30:07  INFO       Saved: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/figures/roc_curves_silver.png
08:30:07  INFO       Saved: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/figures/roc_curves_silver.png
08:30:07  INFO       Saved: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/figures/roc_curves_gold.png
08:30:07  INFO       Saved: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/

In [60]:
# Build explainer & compute SHAP values


if best_model_name in TREE_MODELS:
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer(X)

elif best_model_name == "LogReg":
    explainer   = shap.LinearExplainer(model, X)
    shap_values = explainer(X)

else:
    raise ValueError(
        f"Unknown model name: '{best_model_name}'. "
        f"Expected one of: 'LogReg', 'RandomForest', 'LightGBM', 'XGBoost', 'CatBoost'."
    )


# Plot
plot_shap_beeswarm(
    shap_values = shap_values,
    title       = f"SHAP Beeswarm — {best_model_name}",
    path        = os.path.join(FIGURES_DIR,"shap_beeswarm_xgb.png"),
    max_display = 20,
)

22:16:49  INFO       Saved: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/figures/shap_beeswarm_xgb.png
22:16:49  INFO       Saved: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/figures/shap_beeswarm_xgb.png
22:16:49  INFO       Saved: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/figures/shap_beeswarm_xgb.png


In [61]:
plot_shap_beeswarm(
    shap_values = shap_values,
    title       = f"SHAP Beeswarm — {best_model_name}",
    path        = os.path.join(FIGURES_DIR,"shap_beeswarm_xgb.png"),
    max_display = 20,
)

22:17:18  INFO       Saved: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/figures/shap_beeswarm_xgb.png
22:17:18  INFO       Saved: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/figures/shap_beeswarm_xgb.png
22:17:18  INFO       Saved: /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/figures/shap_beeswarm_xgb.png


In [62]:
FIGURES_DIR

'/Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/figures'

In [63]:
# ── Report ────────────────────────────────────────────────────────────────
log.info("")
log.info("Writing report ...")
write_report(cv_results, holdout, best_fs, best_model_name,
                tuned_auc, best_params, REPORT_PATH)

22:17:18  INFO     
22:17:18  INFO     
22:17:18  INFO     
22:17:18  INFO     Writing report ...
22:17:18  INFO     Writing report ...
22:17:18  INFO     Writing report ...
22:17:18  INFO       Report saved → /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/model_comparison_report.md
22:17:18  INFO       Report saved → /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/model_comparison_report.md
22:17:18  INFO       Report saved → /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/Reports/model_comparison_report.md


In [64]:


# ── Artifacts ─────────────────────────────────────────────────────────────
log.info("")
log.info("Saving artifacts ...")
save_artifacts(tuned_model, best_model_name, best_fs,
                best_params, tuned_auc, holdout)

log.info("")
log.info("Pipeline complete.")
log.info(f"  Best model : {best_model_name} ({best_fs.upper()})")
log.info(f"  Holdout AUC: {holdout[best_fs][best_model_name]['metrics']['auc_roc']:.4f}")


22:17:18  INFO     
22:17:18  INFO     
22:17:18  INFO     
22:17:18  INFO     Saving artifacts ...
22:17:18  INFO     Saving artifacts ...
22:17:18  INFO     Saving artifacts ...
22:17:18  INFO       Model saved → /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/pipeline_artifacts/best_model.pkl
22:17:18  INFO       Model saved → /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/pipeline_artifacts/best_model.pkl
22:17:18  INFO       Model saved → /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/pipeline_artifacts/best_model.pkl
22:17:18  INFO       Metadata saved → /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/pipeline_artifacts/model_metadata.json
22:17:18  INFO       Metadata saved → /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/main_scripts/pip

In [65]:
0.9411


0.9411

In [66]:
best_params

{'n_estimators': 1422,
 'learning_rate': 0.09800155540717399,
 'max_depth': 3,
 'min_child_weight': 7,
 'subsample': 0.8507948030287987,
 'colsample_bytree': 0.6261385095155073,
 'gamma': 2.5122029905565992,
 'reg_alpha': 5.24425726106788e-08,
 'reg_lambda': 7.210495303924496e-07,
 'scale_pos_weight': np.float64(4.72590106097843),
 'eval_metric': 'auc',
 'n_jobs': -1,
 'random_state': 40,
 'verbosity': 0}